# Solutions · 04 · Shortcut design and multicomponent distillation

Worked solutions to the exercises of [notebook 04](../notebooks/04_shortcut_and_multicomponent_distillation.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engsep import distillation as dist
from engsep import vle
from engsep.vle import y_from_alpha
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
names = ["propane", "isobutane", "n-butane", "n-pentane"]
z = np.array([0.30, 0.15, 0.35, 0.20])
alphas = np.array([vle.antoine_pressure(n, 320.0) for n in names]) / vle.antoine_pressure("n-butane", 320.0)

## Exercise 1

Repeat the depropaniser design with the heavy-key recovery in the distillate at 0.1 % and 5 % instead of 1 %. How do N_min, R_min and the number of stages respond? Which specification is the expensive one?

In [3]:
def design(hk_in_D):
    d_frac = np.array([0.99, 0.098, hk_in_D, 0.0])
    D_i = z * d_frac; B_i = z - D_i
    xD, xB = D_i / D_i.sum(), B_i / B_i.sum()
    N_min = dist.fenske(xD[0], xB[0], alphas[0], xD_hk=xD[2], xB_hk=xB[2])
    R_min = dist.underwood(alphas, z, xD, q=1.0)["R_min"]
    return N_min, R_min, dist.gilliland(1.3 * R_min, R_min, N_min)
for hk in (0.001, 0.01, 0.05):
    N_min, R_min, N = design(hk)
    print(f"heavy key {hk:.1%} to the distillate: N_min = {N_min:.1f}, R_min = {R_min:.3f}, N = {N:.1f} at 1.3 R_min")

heavy key 0.1% to the distillate: N_min = 9.3, R_min = 1.174, N = 20.3 at 1.3 R_min
heavy key 1.0% to the distillate: N_min = 7.4, R_min = 1.142, N = 16.5 at 1.3 R_min
heavy key 5.0% to the distillate: N_min = 6.1, R_min = 1.009, N = 13.9 at 1.3 R_min


Tightening the heavy-key leak from 1 % to 0.1 % adds about three minimum stages and four actual ones while
hardly changing the minimum reflux; loosening it to 5 % saves a similar number. Stages are the price of
purity, reflux the price of recovery - and the heavy-key specification is the expensive one here because it
sets the distillate's purity.

## Exercise 2

The cost model uses a fixed tray efficiency. Replace it with O'Connell's correlation at three viscosities (0.2, 0.5, 1.0 mPa s) and find how the optimum reflux ratio shifts when trays are less efficient.

In [4]:
alpha_bt = vle.geometric_mean_alpha(("benzene", "toluene"), ideal=True)
sc = dist.fug(alpha_bt, 0.45, 0.95, 0.05, R_over_Rmin=1.3)
for mu in (0.2, 0.5, 1.0):
    E = dist.oconnell_efficiency(alpha_bt, mu)
    RR = np.linspace(1.05, 3.0, 60)
    total = []
    for r_ in RR:
        R = r_ * sc["R_min"]; N = dist.gilliland(R, sc["R_min"], sc["N_min"])
        total.append(80e3 * N / E / 10 / 1e3 + (R + 1) * 45.0 * 30e3 / 3600 * 8000 * 0.02 / 1e3)
    print(f"viscosity {mu} mPa s -> efficiency {E:.2f}: optimum at R/R_min = {RR[int(np.argmin(total))]:.2f}")

viscosity 0.2 mPa s -> efficiency 0.58: optimum at R/R_min = 1.58
viscosity 0.5 mPa s -> efficiency 0.47: optimum at R/R_min = 1.71
viscosity 1.0 mPa s -> efficiency 0.39: optimum at R/R_min = 1.78


Less efficient trays make stages more expensive, and the optimum shifts to higher reflux: buy steam instead
of steel. The shift is modest because the cost minimum is broad - the same robustness that lets the 1.2-1.5
rule of thumb survive.

## Exercise 3

**Implement it yourself:** write Underwood's equations for a binary (find theta between the two volatilities, then R_min) and compare with the McCabe-Thiele pinch value for benzene-toluene with q = 1.

In [5]:
alpha_bt = vle.geometric_mean_alpha(("benzene", "toluene"), ideal=True)
zF, xD, q = 0.45, 0.95, 1.0
theta = optimize.brentq(lambda th: alpha_bt * zF / (alpha_bt - th) + (1 - zF) / (1 - th) - (1 - q), 1.0 + 1e-9, alpha_bt - 1e-9)
R_min_uw = alpha_bt * xD / (alpha_bt - theta) + (1 - xD) / (1 - theta) - 1
print(f"by hand: theta = {theta:.4f}, R_min = {R_min_uw:.4f}; McCabe-Thiele pinch: {dist.minimum_reflux(lambda x: y_from_alpha(x, alpha_bt), xD, zF)['R_min']:.4f}")

by hand: theta = 1.4868, R_min = 1.2823; McCabe-Thiele pinch: 1.2823


Underwood's root lies between the two volatilities and reproduces the graphical minimum reflux for a binary
with constant α exactly - the two methods are the same physics.